# Togyzkumalak OCR — Colab training (moves + diagram)

Runtime → Change runtime type → **GPU**, then run the cells in order.

Trains both classifiers on on-the-fly synthetic data (nothing to upload):
1. **moves** — 163-class move-cell reader (`11`..`99x` + `empty`)
2. **diagram** — 85-class board-diagram reader (values `0`-`81`, `x`, `-`, `empty`) for the kazan boxes and 2×9 pit grids of the summary strips

At the end you download one `artifacts.zip` with the ONNX models + class sidecars, ready to drop into `models/` of the HuggingFace Space.

In [ ]:
!nvidia-smi

In [ ]:
# Fetch the code at runtime.
# Option A (default): clone from GitHub
!git clone https://github.com/ansarzeinulla/9OCR.git project
%cd project

# Option B: upload a zip named project.zip instead, then:
# from google.colab import files; files.upload()
# !unzip -q project.zip -d project && cd project

In [ ]:
!pip install -q -r requirements-train.txt

In [ ]:
# Build glyph pools at runtime: ARDIS (European handwriting) + EMNIST via
# torchvision (--emnist is needed on Colab because ingredients/ is gitignored;
# digit 0 comes from these pools).
!python scripts/get_glyphs.py --emnist

In [ ]:
# Sanity check both synthesizers before burning GPU time.
!python -m togyz.synth --preview preview_moves.png --task moves
!python -m togyz.synth --preview preview_diagram.png --task diagram
from IPython.display import Image as IPImage, display
display(IPImage('preview_moves.png'))
display(IPImage('preview_diagram.png'))

In [ ]:
# Train the move classifier (~30 epochs x 50k synthetic cells).
!python train.py --task moves --epochs 30 --samples-per-epoch 50000 --batch-size 256 --workers 2

In [ ]:
# Train the unified board-diagram classifier.
!python train.py --task diagram --epochs 30 --samples-per-epoch 50000 --batch-size 256 --workers 2

In [ ]:
# Export both checkpoints to single-file ONNX (+ .classes.json sidecars).
!python scripts/export_onnx.py

In [ ]:
# Bundle the serving artifacts and download.
# In the Space's models/ dir: best.onnx stays best.onnx, the diagram model
# is renamed to diagram.onnx (+ diagram.classes.json).
import shutil, zipfile
from pathlib import Path

out = Path('artifacts'); out.mkdir(exist_ok=True)
shutil.copy('checkpoints/best.onnx', out / 'best.onnx')
shutil.copy('checkpoints/best.classes.json', out / 'best.classes.json')
shutil.copy('checkpoints/diagram/best.onnx', out / 'diagram.onnx')
shutil.copy('checkpoints/diagram/best.classes.json', out / 'diagram.classes.json')
with zipfile.ZipFile('artifacts.zip', 'w') as zf:
    for f in out.iterdir():
        zf.write(f, arcname=f.name)

from google.colab import files
files.download('artifacts.zip')